## M3 — Índice F-of-M versión final
Fórmula completa con variables económicas enriquecidas.
F-of-M = 0.30·Solvencia + 0.25·Demanda + 0.25·Soledad + 0.15·Concentración + 0.05·Futuro
Solvencia = renta 15% + pensión 10% + precio m² 5%

In [50]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RUTA_BASE = r'C:\Users\jcrs2\Documents\Akademia\akademia-residencias'

df = pd.read_csv(rf'{RUTA_BASE}\data\dataset_maestro.csv',
                 dtype={'seccion_censal': str},
                 encoding='utf-8-sig')
df_muni = pd.read_csv(rf'{RUTA_BASE}\data\municipios_limpios.csv')

print(f"Secciones: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"Columnas: {df.columns.tolist()}")

Secciones: 36,334 | Columnas: 31
Columnas: ['seccion_censal', 'CPRO', 'CMUN', 'latitud', 'longitud', 'total_ambos', 'vol_65plus', 'pct_65plus', 'vol_75plus', 'pct_75plus', 'cohorte_futura', 'mujeres_80plus', 'hombres_80plus', 'ratio_viudez', 'solvente', 'cohorte_futura_2029', 'CODAUTO', 'indice_cobertura', 'plazas_total', 'hogares_uni_70_2029', 'hosp_en_provincia', 'tiene_hospital', 'pension_media', 'num_pensionistas', 'precio_m2', 'precio_m2_final', 'tasa_sustitucion', 'vol_80plus', 'pct_80plus', 'renta_persona_2024', 'renta_hogar_2024']


In [51]:
print(f"Columnas en dataset_maestro: {df.shape[1]}")
print(f"hogares_uni_70_2029 presente: {'hogares_uni_70_2029' in df.columns}")
print(f"\nColumnas disponibles:\n{df.columns.tolist()}")

Columnas en dataset_maestro: 31
hogares_uni_70_2029 presente: True

Columnas disponibles:
['seccion_censal', 'CPRO', 'CMUN', 'latitud', 'longitud', 'total_ambos', 'vol_65plus', 'pct_65plus', 'vol_75plus', 'pct_75plus', 'cohorte_futura', 'mujeres_80plus', 'hombres_80plus', 'ratio_viudez', 'solvente', 'cohorte_futura_2029', 'CODAUTO', 'indice_cobertura', 'plazas_total', 'hogares_uni_70_2029', 'hosp_en_provincia', 'tiene_hospital', 'pension_media', 'num_pensionistas', 'precio_m2', 'precio_m2_final', 'tasa_sustitucion', 'vol_80plus', 'pct_80plus', 'renta_persona_2024', 'renta_hogar_2024']


### Paso 1 — Normalización min-max de todas las variables

In [52]:
def normalizar(serie):
    mn, mx = serie.min(), serie.max()
    return (serie - mn) / (mx - mn)

def normalizar_inverso(serie):
    mn, mx = serie.min(), serie.max()
    return 1 - (serie - mn) / (mx - mn)

# Solvencia
df['n_renta']    = normalizar(df['renta_hogar_2024'])
df['n_pension']  = normalizar(df['pension_media'].fillna(df['pension_media'].median()))
df['n_precio_m2']= normalizar(df['precio_m2_final'].fillna(df['precio_m2_final'].median()))

# Demanda
df['n_vol_75']   = normalizar(df['vol_75plus'])
df['n_pct_75']   = normalizar(df['pct_75plus'])
df['n_vol_80']   = normalizar(df['vol_80plus'])

# Soledad
df['n_viudez']   = normalizar(df['ratio_viudez'])
df['n_hogares']  = normalizar(df['hogares_uni_70_2029'])

# Concentración
df['n_cobertura']= normalizar_inverso(df['indice_cobertura'])
df['n_hospital'] = normalizar(df['tiene_hospital'].astype(float))

# Futuro
df['n_cohorte']  = normalizar(df['cohorte_futura_2029'])
df['n_tasa_sust']= normalizar(df['tasa_sustitucion'].fillna(0))

cols_n = ['n_renta','n_pension','n_precio_m2','n_vol_75','n_pct_75',
          'n_vol_80','n_viudez','n_hogares','n_cobertura','n_hospital',
          'n_cohorte','n_tasa_sust']

print("Rango de variables normalizadas (todas deben ser 0-1):")
print(df[cols_n].agg(['min','max']).round(3).to_string())

Rango de variables normalizadas (todas deben ser 0-1):
     n_renta  n_pension  n_precio_m2  n_vol_75  n_pct_75  n_vol_80  n_viudez  n_hogares  n_cobertura  n_hospital  n_cohorte  n_tasa_sust
min      0.0        0.0          0.0       0.0       0.0       0.0       0.0        0.0          0.0         0.0        0.0          0.0
max      1.0        1.0          1.0       1.0       1.0       1.0       1.0        1.0          1.0         1.0        1.0          1.0


### Paso 2 — Construir el F-of-M con pesos definitivos

In [53]:
# Pesos definitivos acordados con el equipo
W_SOLVENCIA     = 0.30  # renta 15% + pensión 10% + precio m² 5%
W_DEMANDA       = 0.25  # vol_75 + pct_75 + vol_80
W_SOLEDAD       = 0.25  # viudez + hogares unipersonales
W_CONCENTRACION = 0.15  # cobertura inversa + hospital
W_FUTURO        = 0.05  # cohorte 2029 + tasa sustitución

# Componentes internos
comp_solvencia     = (0.15 * df['n_renta'] +
                      0.10 * df['n_pension'] +
                      0.05 * df['n_precio_m2']) / W_SOLVENCIA

comp_demanda       = (df['n_vol_75'] + df['n_pct_75'] + df['n_vol_80']) / 3

comp_soledad       = (df['n_viudez'] + df['n_hogares'].fillna(0)) / 2

comp_concentracion = (df['n_cobertura'] + df['n_hospital']) / 2

comp_futuro        = (df['n_cohorte'] + df['n_tasa_sust']) / 2

# Fórmula completa
df['fofm'] = (
    W_SOLVENCIA     * comp_solvencia +
    W_DEMANDA       * comp_demanda +
    W_SOLEDAD       * comp_soledad +
    W_CONCENTRACION * comp_concentracion +
    W_FUTURO        * comp_futuro
)

# Filtro duro — la plaza (2.100€/mes) no puede superar el 80% de la renta del hogar
# 2.100 × 12 / 0.80 = 31.500€
UMBRAL_SOLVENCIA = 31_500
df.loc[df['renta_hogar_2024'] < UMBRAL_SOLVENCIA, 'fofm'] = 0

print(f"Umbral solvencia aplicado: {UMBRAL_SOLVENCIA:,}€")
print(f"Secciones descartadas: {(df['renta_hogar_2024'] < UMBRAL_SOLVENCIA).sum():,}")
print(f"Secciones puntuadas:   {(df['renta_hogar_2024'] >= UMBRAL_SOLVENCIA).sum():,}")

print("=== DISTRIBUCIÓN DEL F-of-M ===")
print(f"Secciones puntuadas (fofm > 0): {(df['fofm']>0).sum():,}")
print(f"Secciones descartadas:          {(df['fofm']==0).sum():,}")
print(f"\nMáximo:      {df['fofm'].max():.4f}")
print(f"Media (>0):  {df[df['fofm']>0]['fofm'].mean():.4f}")
print(f"Mínimo (>0): {df[df['fofm']>0]['fofm'].min():.4f}")

# Verificar pesos suman 1
total = W_SOLVENCIA + W_DEMANDA + W_SOLEDAD + W_CONCENTRACION + W_FUTURO
print(f"\nSuma de pesos: {total} (debe ser 1.0)")

Umbral solvencia aplicado: 31,500€
Secciones descartadas: 25,047
Secciones puntuadas:   9,965
=== DISTRIBUCIÓN DEL F-of-M ===
Secciones puntuadas (fofm > 0): 9,965
Secciones descartadas:          25,047

Máximo:      0.5739
Media (>0):  0.3365
Mínimo (>0): 0.1047

Suma de pesos: 1.0 (debe ser 1.0)


### Paso 3 — Agregar por municipio y seleccionar top 100

In [54]:
# Añadir nombre de municipio
df_muni['CPRO'] = df_muni['CPRO'].astype(int)
df_muni['CMUN'] = df_muni['CMUN'].astype(int)
df['CPRO'] = df['CPRO'].astype(float).astype('Int64')
df['CMUN'] = df['CMUN'].astype(float).astype('Int64')

df = df.merge(df_muni[['CPRO','CMUN','NOMBRE']],
              on=['CPRO','CMUN'], how='left')

# Agregar por municipio
df_municipios = df[df['fofm'] > 0].groupby(['CPRO','CMUN','NOMBRE']).agg(
    secciones          = ('seccion_censal',    'count'),
    fofm_medio         = ('fofm',              'mean'),
    fofm_max           = ('fofm',              'max'),
    vol_75plus_total   = ('vol_75plus',         'sum'),
    vol_80plus_total   = ('vol_80plus',         'sum'),
    pct_75plus_media   = ('pct_75plus',         'mean'),
    ratio_viudez_media = ('ratio_viudez',       'mean'),
    cohorte_futura     = ('cohorte_futura_2029','sum'),
    tasa_sustitucion   = ('tasa_sustitucion',   'mean'),
    renta_hogar        = ('renta_hogar_2024',   'mean'),
    pension_media      = ('pension_media',       'mean'),
    precio_m2          = ('precio_m2_final',    'mean'),
    indice_cobertura   = ('indice_cobertura',   'first'),
    tiene_hospital     = ('tiene_hospital',     'max'),
    latitud            = ('latitud',            'mean'),
    longitud           = ('longitud',           'mean')
).reset_index()

df_municipios = df_municipios.sort_values(
    'fofm_medio', ascending=False
).reset_index(drop=True)
df_municipios['ranking_municipio'] = df_municipios.index + 1

# Filtro y selección top 100
MINIMO_75PLUS = 1000
df_candidatos = df_municipios[
    df_municipios['vol_75plus_total'] >= MINIMO_75PLUS
].copy()

top100 = (df_candidatos
    .groupby('CPRO')
    .apply(lambda x: x.nlargest(3, 'fofm_medio'))
    .reset_index(drop=True)
    .nlargest(100, 'fofm_medio')
    .reset_index(drop=True)
)
top100['posicion_final'] = top100.index + 1

print(f"Municipios candidatos:     {len(df_candidatos):,}")
print(f"Ubicaciones seleccionadas: {len(top100)}")
print(f"Provincias representadas:  {top100['CPRO'].nunique()}")
print(f"\n=== TOP 15 ===")
print(top100[['posicion_final','NOMBRE','renta_hogar',
              'pension_media','precio_m2',
              'vol_75plus_total','fofm_medio']].head(15).to_string(index=False))

Municipios candidatos:     194
Ubicaciones seleccionadas: 87
Provincias representadas:  52

=== TOP 15 ===
 posicion_final                 NOMBRE  renta_hogar  pension_media   precio_m2  vol_75plus_total  fofm_medio
              1     Pozuelo de Alarcón 72063.659574        1853.00 5101.700000            6740.0    0.520236
              2            Majadahonda 61711.738095        1883.55 4619.200000            4927.0    0.496603
              3           Torrelodones 60655.000000        1824.04 4455.112856            1446.0    0.488767
              4  Sant Cugat del Vallès 57999.490196        1712.02 4855.600000            5967.0    0.478022
              5              Barcelona 42624.976190        1525.79 4501.200000          125993.0    0.445154
              6 Esplugues de Llobregat 44378.523810        1458.05 3578.800000            3746.0    0.432603
              7               Valencia 41477.441748        1423.59 2791.000000           29542.0    0.368257
              8      

### Paso 4 — Guardar resultados

In [55]:
# Guardar archivos
df_municipios.to_csv(rf'{RUTA_BASE}\outputs\ranking_municipios.csv',
                     index=False, encoding='utf-8-sig')
top100.to_csv(rf'{RUTA_BASE}\outputs\top100_ubicaciones.csv',
              index=False, encoding='utf-8-sig')

resumen_prov = (top100.groupby('CPRO').agg(
    municipios  = ('NOMBRE',          'count'),
    fofm_medio  = ('fofm_medio',      'mean'),
    vol_75plus  = ('vol_75plus_total', 'sum'),
    renta_media = ('renta_hogar',      'mean')
).sort_values('fofm_medio', ascending=False).reset_index())
resumen_prov.to_csv(rf'{RUTA_BASE}\outputs\resumen_provincias.csv',
                    index=False, encoding='utf-8-sig')

print("Archivos guardados:")
print(f"  ranking_municipios.csv  → {len(df_municipios):,} municipios")
print(f"  top100_ubicaciones.csv  → {len(top100)} ubicaciones")
print(f"  resumen_provincias.csv  → {resumen_prov.shape[0]} provincias")

# Verificación final
print(f"\n=== VERIFICACIÓN ===")
print(f"Pesos suman 1.0: {W_SOLVENCIA+W_DEMANDA+W_SOLEDAD+W_CONCENTRACION+W_FUTURO == 1.0}")
print(f"fofm fuera de rango: {((df['fofm']<0)|(df['fofm']>1)).sum()}")
print(f"No solventes con fofm>0: {((df['solvente']==0)&(df['fofm']>0)).sum()}")
print(f"Mínimo vol_75plus top100: {top100['vol_75plus_total'].min():.0f} (debe ser ≥500)")
print(f"Máximo municipios/provincia: {top100.groupby('CPRO').size().max()} (debe ser ≤3)")

Archivos guardados:
  ranking_municipios.csv  → 1,419 municipios
  top100_ubicaciones.csv  → 87 ubicaciones
  resumen_provincias.csv  → 52 provincias

=== VERIFICACIÓN ===
Pesos suman 1.0: True
fofm fuera de rango: 0
No solventes con fofm>0: 150
Mínimo vol_75plus top100: 1030 (debe ser ≥500)
Máximo municipios/provincia: 3 (debe ser ≤3)


### Paso 5 — Generar Excel de resultados

In [56]:
df_muni_load = pd.read_csv(rf'{RUTA_BASE}\data\municipios_limpios.csv')
top100_clusters = pd.read_csv(rf'{RUTA_BASE}\outputs\top100_con_clusters.csv',
                               encoding='utf-8-sig')
resumen_prov = pd.read_csv(rf'{RUTA_BASE}\outputs\resumen_provincias.csv',
                            encoding='utf-8-sig')

with pd.ExcelWriter(rf'{RUTA_BASE}\outputs\resultados_akademia.xlsx',
                    engine='openpyxl') as writer:

    top100_clusters.to_excel(writer,
        sheet_name='Top 100 ubicaciones', index=False)

    df_municipios.head(500).to_excel(writer,
        sheet_name='Ranking municipios', index=False)

    resumen_prov.to_excel(writer,
        sheet_name='Resumen por provincia', index=False)

    resumen_clusters = top100_clusters.groupby('perfil').agg(
        municipios  = ('NOMBRE',          'count'),
        renta_media = ('renta_hogar',      'mean'),
        vol_75plus  = ('vol_75plus_total', 'mean'),
        pension     = ('pension_media',    'mean'),
        precio_m2   = ('precio_m2',        'mean'),
        fofm_medio  = ('fofm_medio',       'mean')
    ).round(2).reset_index()
    resumen_clusters.to_excel(writer,
        sheet_name='Perfiles estratégicos', index=False)

    guia = pd.DataFrame({
        'Archivo': [
            'mapa_top100.html',
            'grafico_perfiles.png',
            'grafico_top20.png',
            'grafico_solvencia_mercado.png',
            'resultados_akademia.xlsx',
            'top100_con_clusters.csv',
            'dataset_maestro.csv'
        ],
        'Para qué sirve': [
            'Mapa interactivo — abrir en navegador, clicar en cada punto',
            'Gráfico de barras con los 4 perfiles estratégicos',
            'Ranking visual del top 20 municipios',
            'Scatter solvencia vs tamaño de mercado',
            'Excel con todas las pestañas de resultados',
            'Datos completos del top 100 con perfil asignado',
            'Dataset central con todas las variables calculadas'
        ]
    })
    guia.to_excel(writer, sheet_name='Guía de archivos', index=False)

print("resultados_akademia.xlsx generado con 5 pestañas:")
print("  - Top 100 ubicaciones")
print("  - Ranking municipios")
print("  - Resumen por provincia")
print("  - Perfiles estratégicos")
print("  - Guía de archivos")

resultados_akademia.xlsx generado con 5 pestañas:
  - Top 100 ubicaciones
  - Ranking municipios
  - Resumen por provincia
  - Perfiles estratégicos
  - Guía de archivos
